# CIFAKE Real vs AI-Generated Image Training (Google Colab)

Run top to bottom with **Runtime → Change runtime type → T4 GPU**. This workflow trains the Custom CNN from scratch and fine-tunes MobileNetV2 on CIFAKE: `REAL` images map to label 0 and `FAKE` images map to label 1. The dataset layout is expected to be `train/REAL`, `train/FAKE`, `test/REAL`, and `test/FAKE`.

> **Task limitation:** CIFAKE distinguishes CIFAR-10 photographs from AI-generated images; it does not label image edits, splicing, or copy-move tampering. See the [CIFAKE dataset documentation](https://github.com/jordan-bird/CIFAKE-Real-and-AI-Generated-Synthetic-Images). These models are experimental AI-generated-image classifiers, not replacements for the project's CASIA manipulation-detection models. The source archive contains 32×32 images, so the models below use 32×32 inputs and are not compatible with the application's 224×224 model interface.

## Dataset setup

1. The archive is local at `data/raw/archive (2).zip`, but a Colab VM cannot read your computer's project folder directly. Upload that ZIP into `/content` using Colab's **Files** sidebar, or choose it when the upload cell prompts you. Upload only this archive if using the automatic ZIP selection.
2. The archive must contain the four class folders listed above. The notebook extracts to a fresh temporary folder in the Colab VM; it does not change the local archive or any CASIA data.
3. The archive's provided `test/` partition stays held out. The notebook makes a stratified validation split from `train/` and never uses test labels for model selection.

Both models, metrics, and plots are saved under `/content/cifake_training_outputs` and downloaded as a ZIP at the end. Model artifacts are CIFAKE-specific and must not be used by the image-manipulation web app.


## 1. Upload the CIFAKE archive and set its path


In [ ]:
from google.colab import files
from pathlib import Path

# Prefer a ZIP uploaded with the Colab Files sidebar. It should appear in /content.
zip_candidates = sorted(Path("/content").glob("*.zip"))
if len(zip_candidates) == 1:
    DATASET_ZIP = zip_candidates[0]
elif len(zip_candidates) > 1:
    raise ValueError(f"More than one ZIP is in /content; keep only the dataset ZIP or set DATASET_ZIP manually: {zip_candidates}")
else:
    print("No ZIP found in /content. Choose the dataset ZIP in the file picker.")
    uploaded_files = files.upload()
    zip_names = [name for name in uploaded_files if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError(f"Upload exactly one dataset ZIP file; received: {list(uploaded_files)}")
    DATASET_ZIP = Path("/content") / zip_names[0]
    del uploaded_files
if not DATASET_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATASET_ZIP}")
print(f"Dataset archive: {DATASET_ZIP} ({DATASET_ZIP.stat().st_size / (1024**3):.2f} GiB)")


## 2. Get project code, install dependencies, and verify GPU

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_URL = "https://github.com/Salman-ahmed-2/image-manipulation-detection.git"
REPO_DIR = Path("/content/image-manipulation-detection")
if not (REPO_DIR / "config.py").is_file():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))
import importlib.util
required_packages = {
    "pandas": "pandas", "sklearn": "scikit-learn", "PIL": "Pillow",
    "matplotlib": "matplotlib", "dotenv": "python-dotenv",
}
missing_packages = [package for module, package in required_packages.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
if not gpus: raise RuntimeError("No GPU found. In Colab choose Runtime → Change runtime type → GPU, then rerun.")
for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)
print("GPU:", gpus)


## 3. Extract CIFAKE and create train/validation/test partitions


In [ ]:
import tempfile
import zipfile
from pathlib import Path, PurePosixPath

import pandas as pd
from sklearn.model_selection import train_test_split
from config import Config

CLASS_TO_LABEL = {"REAL": 0, "FAKE": 1}
EXPECTED_PARTITIONS = {"train", "test"}
DATASET_DIR = Path(tempfile.mkdtemp(prefix="cifake_dataset_"))

with zipfile.ZipFile(DATASET_ZIP) as archive:
    image_members = [member for member in archive.infolist() if not member.is_dir()]
    member_names = [member.filename for member in image_members]
    if len(member_names) != len(set(member_names)):
        raise ValueError("The CIFAKE archive contains duplicate file paths.")

    for member in image_members:
        member_path = PurePosixPath(member.filename)
        parts = member_path.parts
        if (
            member_path.is_absolute()
            or "\\" in member.filename
            or any(part in {"", ".", ".."} for part in parts)
            or len(parts) != 3
            or parts[0] not in EXPECTED_PARTITIONS
            or parts[1] not in CLASS_TO_LABEL
            or member_path.suffix.lower() != ".jpg"
        ):
            raise ValueError(f"Unexpected path in CIFAKE archive: {member.filename}")
        target = (DATASET_DIR.joinpath(*parts)).resolve()
        if not target.is_relative_to(DATASET_DIR.resolve()):
            raise ValueError(f"Unsafe path in CIFAKE archive: {member.filename}")

    if not image_members:
        raise ValueError("The CIFAKE archive contains no JPEG images.")

    print(f"Extracting {len(image_members):,} images to {DATASET_DIR}...", flush=True)
    for index, member in enumerate(image_members, start=1):
        archive.extract(member, DATASET_DIR)
        if index % 5000 == 0 or index == len(image_members):
            print(f"Extracted {index:,}/{len(image_members):,} images", flush=True)

records = []
for member in image_members:
    parts = PurePosixPath(member.filename).parts
    partition, class_name, filename = parts
    records.append({
        "filepath": str(DATASET_DIR.joinpath(*parts).resolve()),
        "filename": filename,
        "source_partition": partition,
        "class_name": class_name,
        "label": CLASS_TO_LABEL[class_name],
    })
all_df = pd.DataFrame(records)
partition_counts = all_df.groupby(["source_partition", "class_name"]).size()
required_pairs = {(partition, class_name) for partition in EXPECTED_PARTITIONS
                  for class_name in CLASS_TO_LABEL}
if set(partition_counts.index) != required_pairs:
    raise ValueError(f"Each train/test partition must contain REAL and FAKE images: {partition_counts.to_dict()}")

source_train = all_df[all_df["source_partition"] == "train"].reset_index(drop=True)
test_df = all_df[all_df["source_partition"] == "test"].copy()
train_df, val_df = train_test_split(
    source_train,
    test_size=Config.DATASET_VALIDATION_RATIO,
    random_state=Config.RANDOM_SEED,
    stratify=source_train["label"],
)
train_df = train_df.assign(split="train").reset_index(drop=True)
val_df = val_df.assign(split="validation").reset_index(drop=True)
test_df = test_df.assign(split="test").reset_index(drop=True)
split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("Source archive class counts:")
display(partition_counts.rename("images").reset_index())
print("Training partitions (label 0 = REAL; label 1 = FAKE):")
display(pd.crosstab(split_df["split"], split_df["class_name"], margins=True))


## 4. Build streaming TensorFlow input pipelines at the source image resolution


In [ ]:
import numpy as np
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight

CIFAKE_IMAGE_SIZE = (32, 32)
CIFAKE_INPUT_SHAPE = (32, 32, 3)
IMAGE_SIZE = CIFAKE_IMAGE_SIZE
BATCH_SIZE, SEED = Config.BATCH_SIZE, Config.RANDOM_SEED
CNN_EPOCHS = 20
MOBILENET_WARMUP_EPOCHS = 6
MOBILENET_FINE_TUNE_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 4
CLASS_NAMES = ["REAL (CIFAR-10 photo)", "FAKE (AI-generated)"]
OUTPUT_DIR = Path("/content/cifake_training_outputs")
for subdir in ("models", "plots", "metrics"):
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)
CACHE_DIR = DATASET_DIR / "_tf_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)

sample_path = Path(train_df["filepath"].iloc[0])
with Image.open(sample_path) as sample_image:
    if sample_image.size != CIFAKE_IMAGE_SIZE:
        raise ValueError(
            f"Expected CIFAKE images at {CIFAKE_IMAGE_SIZE}, got {sample_image.size} "
            f"for {sample_path}. Update CIFAKE_IMAGE_SIZE only if using a documented variant."
        )

def decode_resize(path_tensor):
    def load_image(path_value):
        with Image.open(path_value.decode("utf-8")) as image:
            image = image.convert("RGB").resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
            return np.asarray(image, dtype=np.uint8)
    image = tf.numpy_function(load_image, [path_tensor], tf.uint8)
    image.set_shape((IMAGE_SIZE[1], IMAGE_SIZE[0], 3))
    return image

def make_dataset(frame, normalization, training=False):
    split_name = str(frame["split"].iloc[0])
    dataset = tf.data.Dataset.from_tensor_slices((
        frame["filepath"].astype(str).to_numpy(),
        frame["label"].to_numpy(dtype=np.int32),
    ))
    dataset = dataset.map(lambda path, label: (decode_resize(path), label),
                          num_parallel_calls=tf.data.AUTOTUNE)
    cache_file = CACHE_DIR / f"{split_name}_32_uint8.cache"
    dataset = dataset.cache(str(cache_file))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    def normalize(image, label):
        image = tf.cast(image, tf.float32) / 255.0
        if normalization == "mobilenet":
            image = image * 2.0 - 1.0
        return image, tf.cast(label, tf.float32)
    return dataset.map(normalize, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

for name, frame in (("train", train_df), ("validation", val_df), ("test", test_df)):
    if frame.empty or set(frame.label.unique()) != {0, 1}:
        raise ValueError(f"{name} split must contain both classes; got {frame.label.value_counts().to_dict()}")
classes = np.array([0, 1])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_df.label.to_numpy())
CLASS_WEIGHTS = {int(label): float(weight) for label, weight in zip(classes, weights)}
cache_gib = len(split_df) * IMAGE_SIZE[0] * IMAGE_SIZE[1] * 3 / (1024**3)
print("Class weights:", CLASS_WEIGHTS)
print(f"Input shape: {CIFAKE_INPUT_SHAPE}; image cache: about {cache_gib:.2f} GiB")
print(f"Split sizes: train={len(train_df):,}, validation={len(val_df):,}, test={len(test_df):,}")


## 5. Shared training and evaluation helpers


In [ ]:
import json
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

def get_callbacks(model_key, patience=EARLY_STOPPING_PATIENCE):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            str(OUTPUT_DIR / "models" / f"{model_key}_best.keras"),
            monitor="val_loss", save_best_only=True, verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience,
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3,
            min_lr=1e-6, verbose=1,
        ),
    ]

def evaluate_split(model, frame, normalization, split_name):
    probabilities = model.predict(make_dataset(frame, normalization), verbose=1).reshape(-1)
    labels = frame.label.to_numpy(dtype=np.int32)
    predictions = (probabilities >= Config.CLASSIFICATION_THRESHOLD).astype(np.int32)
    metrics = {
        "n": int(len(labels)),
        "class_labels": {"0": CLASS_NAMES[0], "1": CLASS_NAMES[1]},
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "f1_score": float(f1_score(labels, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "threshold": float(Config.CLASSIFICATION_THRESHOLD),
        "confusion_matrix_labels_0_1": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
        "classification_report": classification_report(
            labels, predictions, target_names=CLASS_NAMES,
            output_dict=True, zero_division=0,
        ),
    }
    print(f"\n{split_name.upper()} RESULTS — {len(labels):,} images")
    print(classification_report(labels, predictions, target_names=CLASS_NAMES, zero_division=0))
    print(f"Accuracy {metrics['accuracy']:.4f} | Precision {metrics['precision']:.4f} | "
          f"Recall {metrics['recall']:.4f} | F1 {metrics['f1_score']:.4f} | "
          f"ROC-AUC {metrics['roc_auc']:.4f}")
    print("Confusion matrix [[REAL correct, REAL predicted FAKE], "
          "[FAKE predicted REAL, FAKE correct]]:", metrics["confusion_matrix_labels_0_1"])
    return metrics

def save_history(histories, model_key):
    merged = {}
    for history in histories:
        for key, values in history.history.items():
            merged.setdefault(key, []).extend([float(value) for value in values])
    pd.DataFrame(merged).to_csv(OUTPUT_DIR / "metrics" / f"{model_key}_history.csv", index_label="epoch")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for key in ("accuracy", "val_accuracy"):
        if key in merged:
            axes[0].plot(merged[key], label=key)
    for key in ("loss", "val_loss"):
        if key in merged:
            axes[1].plot(merged[key], label=key)
    axes[0].set_title("Accuracy")
    axes[1].set_title("Loss")
    for axis in axes:
        axis.set_xlabel("Epoch")
        axis.legend()
        axis.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "plots" / f"{model_key}_training_curves.png", dpi=150)
    plt.show()


## 6. Train the Custom CNN from scratch on CIFAKE


In [ ]:
from src.models.cnn_model import build_custom_cnn

cnn = build_custom_cnn(input_shape=CIFAKE_INPUT_SHAPE)
cnn_history = cnn.fit(
    make_dataset(train_df, "standard", training=True),
    validation_data=make_dataset(val_df, "standard"),
    epochs=CNN_EPOCHS,
    callbacks=get_callbacks("cifake_custom_cnn"),
    class_weight=CLASS_WEIGHTS,
    verbose=1,
)
cnn = tf.keras.models.load_model(OUTPUT_DIR / "models" / "cifake_custom_cnn_best.keras")
cnn.save(OUTPUT_DIR / "models" / "cifake_custom_cnn.keras")
save_history([cnn_history], "cifake_custom_cnn")


## 7. Train MobileNetV2 from ImageNet weights (do not resume a CASIA checkpoint)


In [ ]:
from src.models.mobilenet_model import build_mobilenet_model, unfreeze_mobilenet_layers

mobile_train = make_dataset(train_df, "mobilenet", training=True)
mobile_val = make_dataset(val_df, "mobilenet")
# CIFAKE has a different target from CASIA; initialize from ImageNet, never a project checkpoint.
mobilenet = build_mobilenet_model(input_shape=CIFAKE_INPUT_SHAPE)
phase1 = mobilenet.fit(
    mobile_train,
    validation_data=mobile_val,
    epochs=MOBILENET_WARMUP_EPOCHS,
    callbacks=get_callbacks("cifake_mobilenetv2_warmup"),
    class_weight=CLASS_WEIGHTS,
    verbose=1,
)
mobilenet = tf.keras.models.load_model(
    OUTPUT_DIR / "models" / "cifake_mobilenetv2_warmup_best.keras"
)
mobilenet = unfreeze_mobilenet_layers(
    mobilenet,
    num_layers_to_unfreeze=Config.MOBILENET_LAYERS_TO_UNFREEZE,
    fine_tune_lr=Config.MOBILENET_FINE_TUNE_LEARNING_RATE,
)
phase2 = mobilenet.fit(
    mobile_train,
    validation_data=mobile_val,
    epochs=MOBILENET_FINE_TUNE_EPOCHS,
    callbacks=get_callbacks("cifake_mobilenetv2_finetune"),
    class_weight=CLASS_WEIGHTS,
    verbose=1,
)
fine_tuned = tf.keras.models.load_model(
    OUTPUT_DIR / "models" / "cifake_mobilenetv2_finetune_best.keras"
)
warmup_model = tf.keras.models.load_model(
    OUTPUT_DIR / "models" / "cifake_mobilenetv2_warmup_best.keras"
)
warmup_val_loss = warmup_model.evaluate(mobile_val, verbose=0)[0]
finetune_val_loss = fine_tuned.evaluate(mobile_val, verbose=0)[0]
print(f"Best warmup validation loss: {warmup_val_loss:.4f}; fine-tuned: {finetune_val_loss:.4f}")
mobilenet = fine_tuned if finetune_val_loss <= warmup_val_loss else warmup_model
mobilenet.save(OUTPUT_DIR / "models" / "cifake_mobilenetv2_best.keras")
mobilenet.save(OUTPUT_DIR / "models" / "cifake_mobilenetv2.keras")
save_history([phase1, phase2], "cifake_mobilenetv2")


## 8. Evaluate validation and held-out test sets (REAL vs AI-generated)


In [ ]:
all_results = {}
for model_key, model, normalization in (
    ("cifake_custom_cnn", cnn, "standard"),
    ("cifake_mobilenetv2", mobilenet, "mobilenet"),
):
    validation_metrics = evaluate_split(model, val_df, normalization, "validation")
    test_metrics = evaluate_split(model, test_df, normalization, "test")
    all_results[model_key] = {"validation": validation_metrics, "test": test_metrics}
    result_path = OUTPUT_DIR / "metrics" / f"{model_key}_results.json"
    result_path.write_text(json.dumps(all_results[model_key], indent=2), encoding="utf-8")
rows = [
    {
        "model": model_key,
        "split": split_name,
        **{key: metrics[key] for key in (
            "n", "accuracy", "precision", "recall", "f1_score", "roc_auc", "threshold"
        )},
    }
    for model_key, split_results in all_results.items()
    for split_name, metrics in split_results.items()
]
results_df = pd.DataFrame(rows)
display(results_df)
results_df.to_csv(OUTPUT_DIR / "metrics" / "validation_test_summary.csv", index=False)


## 9. Download CIFAKE-specific models, plots, and metrics


In [ ]:
import shutil
from google.colab import files

OUTPUT_ZIP = Path("/content/cifake_training_outputs.zip")
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()
shutil.make_archive(str(OUTPUT_ZIP.with_suffix("")), "zip", OUTPUT_DIR)
print(f"Created {OUTPUT_ZIP} ({OUTPUT_ZIP.stat().st_size / (1024**2):.1f} MiB)")
files.download(str(OUTPUT_ZIP))
